# M3 - estatísticas por zona (exploração passo a passo)

**O que o enunciado pede.** Para cada zona de recolha e mês: média e desvio padrão de `fare_amount` e `trip_distance`. Depois, com *window functions*:
1. ranking das zonas por volume de viagens no mês, dentro de cada borough;
2. quota móvel a 3 meses de cada zona dentro do seu borough;
3. alerta para zonas cujo volume do mês se afasta mais de 1,5 desvios padrão da sua própria média móvel.

**Plano do notebook** (um bloco por passo, cada um com `show()` para ver o que mudou):

| Bloco | O quê |
|---|---|
| M3-1 | juntar zona e borough às viagens |
| M3-2 | agregar: uma linha por (zona, mês) |
| M3-3 | completar a grelha zona × mês (meses sem viagens = 0) |
| M3-4 | índice de mês contínuo (`m_idx`) |
| M3-5 | window 1: ranking no borough |
| M3-6 | windows 2 e 3: quota móvel a 3 meses |
| M3-7 | window 4: média móvel e alerta de 1,5σ |
| M3-8 | plano de execução e escrita |
| final | tudo junto numa função `run()` igual às do M1/M2 |

As viagens limpas vêm do `m1_clean.py`, como no `m2-explorar.ipynb`. Uso **6 meses** de 2019: as janelas de 3 meses só mostram alguma coisa com vários meses.

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
         .appName("m3-dev")
         .config("spark.driver.memory", "1g")
         .config("spark.executor.memory", "1200m")
         .getOrCreate())

In [ ]:
import sys, contextlib
from types import SimpleNamespace
from pyspark.sql import Window, functions as F
sys.path.insert(0, "/project/src")
import taxi.m1_clean as m1

class NoProf:                      # profiler que não mede nada (o verdadeiro só no main.py)
    def profile(self, name):
        return contextlib.nullcontext()

a = SimpleNamespace(input="/project/data/raw", zones="/project/data/taxi_zone_lookup.csv",
                    output="/project/output", years=[2019], months="1-6", fraction=1.0)
trips, zones = m1.run(spark, a, set(), NoProf())
trips = trips.cache()      # só no notebook: vamos fazer muitos show() e não queremos reler os Parquet de cada vez
print("viagens limpas:", trips.count())

## Bloco M3-1: zona e borough de recolha

As viagens só têm o número da zona (`PULocationID`). O nome da zona e o borough vêm da tabela de zonas, com o mesmo join do M2. A diferença é que aqui também trazemos `Zone`, para o resultado ser legível.

`LocationID` é renomeado para `PULocationID` para o join ser pelo nome da coluna, sem a ficar duplicada.

In [ ]:
pu = zones.select(F.col("LocationID").alias("PULocationID"),
                  F.col("Zone").alias("zone"), F.col("Borough").alias("borough"))
t = trips.join(pu, "PULocationID")

t.select("PULocationID", "zone", "borough", "year", "month", "fare_amount", "trip_distance").show(5, truncate=False)

## Bloco M3-2: uma linha por (zona, mês)

`groupBy` por zona e mês, com 5 agregados:

| Coluna | Função | Nota |
|---|---|---|
| `trip_count` | `count("*")` | volume da zona no mês, usado em todas as windows |
| `mean_fare`, `mean_distance` | `avg` | |
| `std_fare`, `std_distance` | `stddev` | desvio **amostral** (divide por n−1); com 1 só viagem dá `null` |

`year` e `month` vêm do **nome do ficheiro** (o `m1_clean.load` acrescenta-os com `F.lit`), e não do timestamp. Os dados têm timestamps com datas impossíveis (anos como 2008 ou 2088). Se os usássemos, apareceriam "meses" falsos com meia dúzia de viagens, e esses meses estragariam as janelas.

`borough` e `zone` entram no `groupBy` só para os levar até ao resultado. Como cada `PULocationID` tem um só borough e uma só zona, não mudam os grupos.

A partir daqui os dados são pequenos: no máximo 263 zonas × 6 meses ≈ 1600 linhas. Os passos seguintes, incluindo as windows, já não trabalham sobre milhões de viagens.

In [ ]:
z = (t.groupBy("borough", "PULocationID", "zone", "year", "month")
      .agg(F.count("*").alias("trip_count"),
           F.avg("fare_amount").alias("mean_fare"),
           F.stddev("fare_amount").alias("std_fare"),
           F.avg("trip_distance").alias("mean_distance"),
           F.stddev("trip_distance").alias("std_distance")))

print("linhas (zona, mês):", z.count())
z.orderBy(F.desc("trip_count")).show(5, truncate=False)

## Bloco M3-3: grelha completa zona × mês

**O problema.** Uma zona sem viagens num mês **não tem linha** nesse mês, porque o `groupBy` só cria grupos que existem. Isto traz dois efeitos:
- uma zona que passa de 100 viagens para 0 nunca é avaliada nesse mês, e por isso nunca é marcada como anomalia, apesar de ser a maior anomalia possível;
- nas médias móveis, o mês vazio fica "ausente" em vez de contar como 0.

**A solução.** Construímos todas as combinações (zona, mês), fazemos um `left join` com `z` e pomos `trip_count = 0` onde não havia viagens:
- `zonas`: as 263 zonas válidas, vindas da tabela de zonas (os mesmos IDs que o M1 aceita, de 1 a 263);
- `meses`: os meses pedidos em `a.years`/`a.months`, com a mesma função `months()` do M1;
- `crossJoin`: produto cartesiano. Aqui é barato, porque as duas tabelas são minúsculas (263 × 6).

**Porque não tirar as zonas e os meses do próprio `z`?** Daria o mesmo resultado, mas `z` seria usado 3 vezes (zonas, meses e join). Como não está em cache, o Spark voltaria a ler e agregar as viagens 3 vezes, o que se vê no `explain()` com 3 leituras. Construir a grelha a partir da tabela de zonas e dos argumentos não lê nenhuma viagem.

Uma zona sem nenhuma viagem em todo o período fica com `trip_count = 0` em todos os meses: aparece no fim do ranking, com quota 0 e sem alerta.

As médias e desvios desses meses ficam `null`, o que está certo: sem viagens não há tarifa média.

**Atenção às divisões.** O Spark 4 tem o modo ANSI ligado por defeito, e por isso dividir por 0 **dá erro** (`DIVIDE_BY_ZERO`) em vez de `null`. Com meses a 0 isto pode acontecer, por isso as divisões seguintes usam `F.try_divide`, que devolve `null`.

In [ ]:
zonas = pu.filter(F.col("PULocationID").between(1, 263))
meses = spark.createDataFrame([(y, m) for y in a.years for m in m1.months(a.months)], "year INT, month INT")
grelha = zonas.crossJoin(meses)

print("linhas antes:", z.count(), "| grelha completa:", grelha.count())

z = (grelha.join(z.drop("borough", "zone"), ["PULocationID", "year", "month"], "left")
           .fillna(0, subset=["trip_count"]))

z.filter(F.col("trip_count") == 0).show(5, truncate=False)   # os pares (zona, mês) que faltavam

## Bloco M3-4: índice de mês contínuo (`m_idx`)

As janelas "3 meses para trás" precisam de um número que cresça 1 por mês, **também na mudança de ano**: `m_idx = year * 12 + month`.

Assim dezembro de 2018 dá 24228 e janeiro de 2019 dá 24229, que são meses seguidos. Com `month` sozinho, janeiro (1) ficaria "antes" de dezembro (12).

In [ ]:
z = z.withColumn("m_idx", F.col("year") * 12 + F.col("month"))
z.select("year", "month", "m_idx").distinct().orderBy("m_idx").show()

## Bloco M3-5: window 1, ranking no borough

Uma *window function* calcula um valor para cada linha olhando para um grupo de linhas relacionadas (a "janela"), **sem juntar as linhas numa só**, ao contrário do `groupBy`.

É o equivalente DataFrame do `RANK() OVER (PARTITION BY ... ORDER BY ... DESC)` que aparece nos slides de Spark SQL:

```python
Window.partitionBy("borough", "year", "month")   # uma janela por borough e mês
      .orderBy(F.desc("trip_count"))             # a zona com mais viagens fica em 1º
```

Há três funções de ranking possíveis:

| Função | Empate entre 2º e 3º | Resultado |
|---|---|---|
| `rank` | mesma posição, salta a seguinte | 1, 2, 2, 4 |
| `dense_rank` | mesma posição, não salta | 1, 2, 2, 3 |
| `row_number` | desempata arbitrariamente | 1, 2, 3, 4 |

Usamos `rank`, que é a mesma do slide.

In [ ]:
by_month = Window.partitionBy("borough", "year", "month").orderBy(F.desc("trip_count"))
z = z.withColumn("rank_in_borough", F.rank().over(by_month))

(z.filter((F.col("borough") == "Manhattan") & (F.col("month") == 1))
  .orderBy("rank_in_borough")
  .select("zone", "trip_count", "rank_in_borough")
  .show(5, truncate=False))

## Bloco M3-6: windows 2 e 3, quota móvel a 3 meses

`share_3m` é igual às viagens da zona nos últimos 3 meses (incluindo o atual) a dividir pelas viagens do **borough inteiro** nesses 3 meses.

- **Numerador.** É uma janela por zona que vai do mês atual até 2 meses para trás: `Window.partitionBy("PULocationID").orderBy("m_idx").rangeBetween(-2, 0)`.
- **Denominador.** É uma janela por borough com o mesmo intervalo de meses. Cada linha é um par (zona, mês), por isso somar o `trip_count` de **todas** as linhas do borough com `m_idx` nesse intervalo dá o total do borough nos 3 meses. Não é preciso uma agregação à parte.

**`rangeBetween` ou `rowsBetween`?**
- `rowsBetween(-2, 0)` conta 2 **linhas** para trás.
- `rangeBetween(-2, 0)` conta 2 **unidades de `m_idx`** para trás, ou seja, 2 meses.

Depois da grelha do bloco M3-3 as duas dão o mesmo, porque já não faltam meses. `rangeBetween` é mais seguro: continua certo mesmo que falte uma linha.

**Nota:** nos 2 primeiros meses carregados (janeiro e fevereiro) a janela tem menos de 3 meses, porque os anteriores não foram lidos. A divisão continua coerente, mas convém dizê-lo no relatório.

In [ ]:
zone_3m = Window.partitionBy("PULocationID").orderBy("m_idx").rangeBetween(-2, 0)
borough_3m = Window.partitionBy("borough").orderBy("m_idx").rangeBetween(-2, 0)

z = (z.withColumn("zone_3m", F.sum("trip_count").over(zone_3m))
      .withColumn("borough_3m", F.sum("trip_count").over(borough_3m))
      .withColumn("share_3m", F.try_divide("zone_3m", "borough_3m")))   # try_divide: 0/0 dá null em vez de erro

# uma zona ao longo dos meses (161 = Midtown Center, Manhattan)
(z.filter(F.col("PULocationID") == 161).orderBy("m_idx")
  .select("zone", "month", "trip_count", "zone_3m", "borough_3m", "share_3m").show(truncate=False))

**Verificação.** Em cada borough e mês, as quotas de todas as zonas têm de somar **1**: juntas, as zonas são o borough inteiro. Se não somarem 1, alguma coisa está mal.

In [ ]:
(z.groupBy("borough", "year", "month").agg(F.round(F.sum("share_3m"), 6).alias("soma_quotas"))
  .orderBy("borough", "month").show(40))

## Bloco M3-7: window 4, média móvel e alerta de 1,5σ

Para cada zona e mês queremos saber se o volume do mês se afasta muito do que é habitual **para essa zona**:

```
volume_anomaly = |trip_count − rolling_mean| > 1.5 × rolling_std
```

**Porque é que a média móvel exclui o mês atual (`rangeBetween(-3, -1)`).** Se a janela fosse `(-2, 0)` (o mês atual e os 2 anteriores, 3 valores), o próprio mês puxaria a média e o desvio para si. Com n valores e desvio amostral, o máximo de `|x − média| / desvio` é `(n−1)/√n`. Para n = 3 isso dá **2/√3 ≈ 1,155**, que nunca passa de 1,5, por isso o alerta **nunca disparava**. Com os 3 meses **anteriores**, o mês atual é comparado com o histórico, que é o que se pretende.

**Primeiros meses.** Em janeiro não há meses anteriores, por isso média e desvio são `null`. Em fevereiro há 1 valor, a média existe mas o desvio amostral é `null`. A comparação com `null` dá `null`, e passamos isso a `False` com `coalesce`, como no M1.

In [ ]:
zone_prev = Window.partitionBy("PULocationID").orderBy("m_idx").rangeBetween(-3, -1)

z = (z.withColumn("rolling_mean", F.avg("trip_count").over(zone_prev))
      .withColumn("rolling_std", F.stddev("trip_count").over(zone_prev))
      .withColumn("volume_anomaly", F.coalesce(
          F.abs(F.col("trip_count") - F.col("rolling_mean")) > 1.5 * F.col("rolling_std"),
          F.lit(False))))

# quantas anomalias por mês
z.groupBy("month").agg(F.sum(F.col("volume_anomaly").cast("int")).alias("anomalias")).orderBy("month").show()

In [ ]:
# exemplos: as anomalias com maior desvio, em número de σ
(z.filter("volume_anomaly")
  .withColumn("n_sigmas", F.try_divide(F.abs(F.col("trip_count") - F.col("rolling_mean")), F.col("rolling_std")))
  .orderBy(F.desc("n_sigmas"))
  .select("borough", "zone", "month", "trip_count", "rolling_mean", "rolling_std", "n_sigmas")
  .show(10, truncate=False))

### Cuidado: zonas com pouco volume

Com só 3 meses de histórico, o σ é muito instável. Numa zona com 10, 10 e 10 viagens o desvio é **0**, e qualquer variação (por exemplo 11) já passa a anomalia. Isto acontece sobretudo em zonas com poucas viagens (Staten Island, EWR).

A célula seguinte mostra quantas anomalias vêm desses casos. **Decisão para o grupo:** aceitar e documentar, ou exigir um volume mínimo (por exemplo `rolling_mean >= 30`) para avaliar a zona.

In [ ]:
(z.filter("volume_anomaly")
  .groupBy((F.col("rolling_std") == 0).alias("std_zero"),
           (F.col("rolling_mean") < 30).alias("pouco_volume"))
  .count().show())

## Bloco M3-8: plano de execução e escrita

No plano físico, procurar:
- **`Exchange hashpartitioning(...)`**: cada um é um shuffle. Há um para o `groupBy` do bloco M3-2 e, em princípio, um por cada `partitionBy` diferente das windows: `(borough, year, month)`, `PULocationID` e `borough`.
- **`Window`** e **`Sort`**: cada window ordena os dados da sua partição.
- **`BroadcastHashJoin`** ou **`SortMergeJoin`** no join com as zonas e no da grelha.

Para a fase de otimização, é aqui que se vê o custo das windows. Mudar a ordem das windows ou repartir logo por `borough` pode poupar shuffles, e isso mede-se depois com o profiler.

In [ ]:
res = z.drop("m_idx", "zone_3m", "borough_3m")
res.explain()

In [ ]:
res.write.mode("overwrite").partitionBy("borough").parquet("/project/output/m3")
spark.read.parquet("/project/output/m3").show(5, truncate=False)

## Versão final: tudo junto

Os blocos acima numa só função, com a mesma assinatura do M2 (`run(trips, zones, a, opt)`). É isto que iria para `src/taxi/m3_zones.py` e que o `main.py` já sabe chamar (`"M3": "m3_zones"` em `OPS`). No ficheiro, `m1` seria `from taxi import m1_clean as m1`.

In [ ]:
def run(trips, zones, a, opt):
    # 1) zona e borough de recolha
    pu = zones.select(F.col("LocationID").alias("PULocationID"),
                      F.col("Zone").alias("zone"), F.col("Borough").alias("borough"))
    t = trips.join(pu, "PULocationID")

    # 2) uma linha por (zona, mês). year/month vêm do nome do ficheiro (m1_clean.load)
    z = (t.groupBy("borough", "PULocationID", "zone", "year", "month")
          .agg(F.count("*").alias("trip_count"),
               F.avg("fare_amount").alias("mean_fare"),
               F.stddev("fare_amount").alias("std_fare"),
               F.avg("trip_distance").alias("mean_distance"),
               F.stddev("trip_distance").alias("std_distance")))

    # 3) grelha zona x mês: mês sem viagens = 0 (senão a zona nem é avaliada nesse mês).
    #    Feita a partir das zonas e dos argumentos: tirá-la de z faria o Spark ler as viagens 3x.
    meses = trips.sparkSession.createDataFrame(
        [(y, m) for y in a.years for m in m1.months(a.months)], "year INT, month INT")
    grelha = pu.filter(F.col("PULocationID").between(1, 263)).crossJoin(meses)
    z = (grelha.join(z.drop("borough", "zone"), ["PULocationID", "year", "month"], "left")
               .fillna(0, subset=["trip_count"])
               .withColumn("m_idx", F.col("year") * 12 + F.col("month")))   # meses seguidos entre anos

    # 4) windows
    by_month = Window.partitionBy("borough", "year", "month").orderBy(F.desc("trip_count"))
    zone_3m = Window.partitionBy("PULocationID").orderBy("m_idx").rangeBetween(-2, 0)
    borough_3m = Window.partitionBy("borough").orderBy("m_idx").rangeBetween(-2, 0)
    # mês atual fora da média: com ele lá dentro e só 3 valores, |desvio| nunca passa 1.15σ
    zone_prev = Window.partitionBy("PULocationID").orderBy("m_idx").rangeBetween(-3, -1)

    res = (z.withColumn("rank_in_borough", F.rank().over(by_month))
            # try_divide: borough sem viagens nos 3 meses dá null em vez de erro (Spark 4 tem ANSI ligado)
            .withColumn("share_3m", F.try_divide(F.sum("trip_count").over(zone_3m),
                                                 F.sum("trip_count").over(borough_3m)))
            .withColumn("rolling_mean", F.avg("trip_count").over(zone_prev))
            .withColumn("rolling_std", F.stddev("trip_count").over(zone_prev))
            # null (menos de 2 meses anteriores) -> False, como no M1
            .withColumn("volume_anomaly", F.coalesce(
                F.abs(F.col("trip_count") - F.col("rolling_mean")) > 1.5 * F.col("rolling_std"),
                F.lit(False)))
            .drop("m_idx"))

    # 5) escrever: uma pasta por borough (como o M2)
    res.write.mode("overwrite").partitionBy("borough").parquet(f"{a.output}/m3")
    return res

final = run(trips, zones, a, set())
final.orderBy("borough", "PULocationID", "year", "month").show(5, truncate=False)

In [ ]:
trips.unpersist()